# DepthWizard — GAMUS metric fine-tune (Colab T4)

Trains Depth Anything V2 to predict **height in metres** instead of a relative field.

## Why this run exists

Blind — no fitting to ground truth anywhere — the pipeline measures **RMSE 16.07 m / MAE 10.33 m**.
The shape is broadly fine (correlation 0.36–0.53); a truth-fitted scale brings the *same*
predictions to about 3.9 m. The error is almost entirely **scale**: the pipeline emits
71–90 m/unit on every tile while each tile actually needs 17–57, because a relative depth
field forces it to assume one ("tallest structure ≈ 40 m").

Two other routes to scale were measured and are dead on this data:

- **Shadow calibration** — 0–6 usable shadows per tile against the ~10 it needs.
- **A 2-parameter DEM fit** — SRTM relief over these tiles is 1–6 m and SRTM is quantised
  to 1 m, so fitted slopes came out 0.083–0.572 where they should be ~1.0.

A model that predicts metres directly removes the guess rather than improving it.

## Two gates

1. **Beat the baseline.** The stock backbone is scored *with* a best-fit affine; this model
   is scored *without* one. That asymmetry favours the baseline deliberately. Baseline to
   beat: **MAE 4.28 m, correlation 0.299**.
2. **Improve the blind benchmark.** Run back on the laptop, because it needs the DFC2019
   data. A model that wins gate 1 and not gate 2 improved the model but not the product.

Only a run that passes gate 1 writes `metric_ok.json`, and only that marker makes the
pipeline use the weights. A losing run leaves everything untouched.

## 1 · Check the GPU

Runtime → Change runtime type → **T4 GPU**. If this prints CPU, fix it before going on —
a CPU run here takes about a week.

In [ ]:
import torch, subprocess
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU.')

## 2 · Upload the two scripts

`train_height.py` imports only `gamus.py` — nothing else from the repo — so these two
files are the whole dependency. Upload both from your local `depthwizard/` folder.

In [ ]:
from google.colab import files
import os
need = ['gamus.py', 'train_height.py']
missing = [f for f in need if not os.path.exists(f)]
if missing:
    print('upload:', missing)
    files.upload()
print('present:', [f for f in need if os.path.exists(f)])

In [ ]:
!pip -q install h5py transformers safetensors

## 3 · Download a GAMUS subset

GAMUS is 80 GB in full; every tile is a separate file, so a subset is enough to train on.
400 training tiles is roughly 4 GB and about 8% of the split.

**Colab disk is ephemeral** — this downloads again on every fresh session.

In [ ]:
!python gamus.py download --split train --limit 400
!python gamus.py download --split val --limit 40

## 4 · Train

~20–40 min on a T4. If you hit CUDA out-of-memory, use `--batch 2 --accum 4` — same
effective batch, half the peak memory.

Watch the last block of output: it prints the baseline and the fine-tuned result side by
side and says plainly whether the checkpoint earned its licence.

In [ ]:
!python train_height.py --epochs 12 --batch 4 --accum 2

## 5 · Did it earn the licence?

`metric_ok.json` is written only when the fine-tuned model beat a baseline that was handed
the optimal scale. If it is absent, **stop here** — the weights are not worth downloading
and the pipeline would refuse them anyway.

In [ ]:
import json, os
marker = 'checkpoints/height_best/metric_ok.json'
if os.path.exists(marker):
    info = json.load(open(marker))
    print('LICENSED\n')
    for k in ('finetuned_mae_m_no_affine', 'baseline_mae_m_with_affine',
              'finetuned_corr', 'baseline_corr', 'epochs', 'trained_tiles'):
        print(f'  {k:<28} {info.get(k)}')
else:
    print('NOT LICENSED — the fine-tuned model did not beat the baseline.')
    print('Options: more tiles (--limit 1200), more epochs, or accept that')
    print('this approach did not pay off and say so. Do not download these weights.')
    if os.path.exists('checkpoints/training_report.json'):
        r = json.load(open('checkpoints/training_report.json'))
        print('\nbaseline:', r.get('baseline_with_affine'))
        print('last epoch:', r.get('history', [{}])[-1])

## 6 · Pull the checkpoint down

About 390 MB. Colab's storage vanishes with the session, so do this before closing the tab.

Unzip into your local `depthwizard/checkpoints/` so the path is
`checkpoints/height_best/` — that is where `DepthBackbone.METRIC_CKPT` looks.

In [ ]:
import os, shutil
from google.colab import files
if os.path.exists('checkpoints/height_best/metric_ok.json'):
    shutil.make_archive('height_best', 'zip', 'checkpoints', 'height_best')
    print('size:', round(os.path.getsize('height_best.zip')/1e6), 'MB')
    files.download('height_best.zip')
else:
    print('Not licensed — nothing to download.')

## 7 · Back on the laptop — gate 2

```bash
# unzip so you have checkpoints/height_best/{config.json, model.safetensors,
#                                            preprocessor_config.json, metric_ok.json}
python -c "from depth_model import DepthBackbone as D; print(D.metric_available())"
python scripts/benchmark_blind.py 10
```

Compare against the current blind figure: **RMSE 16.07 m / MAE 10.33 m**.

Builds will now report tier `B (metric depth model)` and skip the scale assumption
entirely. If a scene's predicted heights come out implausible, the build says so rather
than silently producing an empty city — that guard was added after an under-trained
checkpoint produced 0 prisms with no error.

**If gate 2 does not improve**, the model got better and the product did not. That is a
real result and worth reporting as one.